<a href="https://colab.research.google.com/github/ravitkurakula/RaviGPT/blob/main/RaviGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

In [2]:
with open("/content/drive/MyDrive/RaviGPT/data/tiny-shakespeare.txt", 'r', encoding = "utf-8") as f :
  text = f.read()

  print(len(text))
  print(text[:500])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [3]:
class CharTokenizer :
  def __init__(self,text) :
    self.char = sorted(list(set(text)))
    self.stoi = {ch : i for i,ch in enumerate(self.char)}
    self.itos = {i : ch for i,ch in enumerate(self.char)}

  @property
  def vocab_size(self):
    return len(self.char)

  def encode(self,text) :
    return [self.stoi[ch] for ch in text]

  def decode(self,tokens):
    return ''.join(self.itos[token] for token in tokens)

In [4]:
tokenizer = CharTokenizer(text)

In [5]:
class TextDataset :
  def __init__(self,data,block_size):
    self.data = data
    self.block_size = block_size

  def __len__(self):
    return len(self.data) - self.block_size

  def __getitem__(self,idx):
    x = self.data[idx : idx+self.block_size]
    y = self.data[idx+1 : idx+self.block_size+1]

    return x,y

In [6]:
data = tokenizer.encode(text)
dataset = TextDataset(data,block_size=64)


In [7]:
block_size = 64
batch_size = 32
d_model = 128
n_heads = 4
n_layers = 4
dropout = 0.1

vocab_size  = tokenizer.vocab_size

In [8]:
n = int(0.9 *len(data))

train_data = torch.tensor(data[:n],dtype= torch.long)
val_data = torch.tensor(data[n:],dtype=torch.long)
print("Total : ", len(data))
print("Train", len(train_data))
print("Validation : ", len(val_data))

Total :  1115394
Train 1003854
Validation :  111540


In [9]:
def get_batch(split,batch_size=32,block_size=64):
  data = train_data if split == "train" else val_data

  ix =  torch.randint(len(data)-block_size, (batch_size,))
  x=  torch.stack([data[i:i + block_size] for i in ix])
  y = torch.stack([data[i+1 : i +block_size+1] for i in ix])

  return x,y

In [10]:
x,y = get_batch("train")

print(x.shape)
print(y.shape)

torch.Size([32, 64])
torch.Size([32, 64])


In [11]:
mask = torch.tril(torch.ones(4,4))

In [19]:
class MultiHeadAttention(nn.Module):
  def __init__(self,d_model,num_heads):
    super().__init__()

    self.d_model = d_model
    self.num_heads = num_heads
    self.head_dim = d_model // num_heads

    self.W_Q = nn.Linear(d_model,d_model)
    self.W_K = nn.Linear(d_model,d_model)
    self.W_V = nn.Linear(d_model,d_model)

    self.out_proj  = nn.Linear(d_model,d_model)

  def forward(self,x,mask):
    B,T,C = x.shape

    q = self.W_Q(x)
    k = self.W_K(x)
    v = self.W_V(x)

    q= q.view(B,T,self.num_heads,self.head_dim)
    k= k.view(B,T,self.num_heads,self.head_dim)
    v= v.view(B,T,self.num_heads,self.head_dim)

    q= q.transpose(1,2)
    k= k.transpose(1,2)
    v= v.transpose(1,2)

    scores = q@k.transpose(-2,-1)
    scores = scores/(self.head_dim**0.5)

    scores = scores.masked_fill(mask == 0, float('-inf'))
    attention_weights = torch.softmax(scores,dim=-1)
    output = attention_weights@v

    output = output.transpose(1,2)
    output = output.contiguous().view(B,T,self.d_model)

    output = self.out_proj(output)

    return output